<a href="https://colab.research.google.com/github/Mengo2414/GomezMatiasPython/blob/main/G%C3%B3mez_Castellano_Mat%C3%ADas_Ejercicio_1_sensores_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Limpiar un registro

Cada registro es un diccionario. `.get()` lee un dato sin que el programa se interrumpa si falta. `.strip()` quita espacios de los extremos.

In [ ]:
def limpiar_registro_sensor(registro_raw):
    registro_limpio = {}

    identificador = registro_raw.get("sensor_id", "")
    registro_limpio["sensor_id"] = str(identificador).strip()

    temperatura_raw = registro_raw.get("temperatura", 0.0)
    if isinstance(temperatura_raw, (int, float)):
        temperatura = float(temperatura_raw)
    else:
        texto = str(temperatura_raw)
        texto = texto.replace("°", "").replace("C", "").replace("c", "")
        texto = texto.replace("F", "").replace("f", "").replace(" ", "")
        try:
            temperatura = float(texto)
        except ValueError:
            temperatura = 0.0
    registro_limpio["temperatura"] = temperatura

    ubicacion = registro_raw.get("ubicacion")
    if isinstance(ubicacion, str) and ubicacion.strip():
        registro_limpio["ubicacion"] = ubicacion.strip()
    else:
        registro_limpio["ubicacion"] = "Área No Asignada"

    return registro_limpio

### Qué hace la función

1. Crea un diccionario nuevo para no cambiar el original.
2. Limpia el identificador.
3. Si la temperatura ya es número, la convierte a `float`. Si es texto, le saca unidad y espacios.
4. `try/except` evita que una lectura dañada detenga el programa: en ese caso usa `0.0`.
5. Si la ubicación falta o está vacía, pone `Área No Asignada`.
6. Devuelve el registro nuevo.

In [ ]:
registro_prueba = {
    "sensor_id": " SENSOR-001 ",
    "temperatura": "24.5 °C",
    "ubicacion": " Planta Norte "
}

registro_limpio = limpiar_registro_sensor(registro_prueba)
print(registro_limpio)
print("El original sigue igual:", registro_prueba)

{'sensor_id': 'SENSOR-001', 'temperatura': 24.5, 'ubicacion': 'Planta Norte'}
El original sigue igual: {'sensor_id': ' SENSOR-001 ', 'temperatura': '24.5 °C', 'ubicacion': ' Planta Norte '}


## 2. Resumir sensores

Un bucle `for` visita cada sensor, suma su temperatura y agrega a la lista de alertas solo los que superan el umbral. La condición es estricta: `>`.

In [ ]:
def generar_resumen_sensores(lista_sensores_limpios, umbral_temperatura):
    if not lista_sensores_limpios:
        return [], 0.0

    suma_temperaturas = 0.0
    sensores_en_alerta = []

    for sensor in lista_sensores_limpios:
        temperatura = sensor["temperatura"]
        suma_temperaturas = suma_temperaturas + temperatura
        if temperatura > umbral_temperatura:
            sensores_en_alerta.append(sensor["sensor_id"])

    promedio = round(suma_temperaturas / len(lista_sensores_limpios), 2)
    return sensores_en_alerta, promedio

La función devuelve una **tupla** con dos elementos: primero las alertas y después el promedio.

In [ ]:
registros_crudos = [
    {"sensor_id": " SENSOR-001 ", "temperatura": "24.5C", "ubicacion": " Planta Norte "},
    {"sensor_id": "SENSOR-002", "temperatura": "35.0 C", "ubicacion": ""},
    {"sensor_id": " SENSOR-003 ", "temperatura": 29, "ubicacion": "Sala de Calderas"},
    {"sensor_id": "SENSOR-004", "temperatura": "dato corrupto", "ubicacion": None},
]

sensores_limpios = [limpiar_registro_sensor(registro) for registro in registros_crudos]
alertas, promedio = generar_resumen_sensores(sensores_limpios, 30.0)

print("Sensores limpios:")
for sensor in sensores_limpios:
    print(sensor)
print("Alertas:", alertas)
print("Promedio:", promedio)

Sensores limpios:
{'sensor_id': 'SENSOR-001', 'temperatura': 24.5, 'ubicacion': 'Planta Norte'}
{'sensor_id': 'SENSOR-002', 'temperatura': 35.0, 'ubicacion': 'Área No Asignada'}
{'sensor_id': 'SENSOR-003', 'temperatura': 29.0, 'ubicacion': 'Sala de Calderas'}
{'sensor_id': 'SENSOR-004', 'temperatura': 0.0, 'ubicacion': 'Área No Asignada'}
Alertas: ['SENSOR-002']
Promedio: 22.12


### Cómo leer el resultado del ejemplo con sensores

En el ejemplo anterior, `SENSOR-002` tiene `35.0 °C` y el umbral es `30.0 °C`. Como 35 es mayor que 30, la lista de alertas debe mostrar `['SENSOR-002']`. El promedio esperado es `22.12 °C`.

### Prueba adicional: lista vacía

Esta es otra prueba distinta. Como no hay sensores, no se puede calcular un promedio; por eso la función devuelve `([], 0.0)` y evita una división por cero.

In [ ]:
resultado_sin_sensores = generar_resumen_sensores([], 30.0)
print("Resultado de la prueba con lista vacía:", resultado_sin_sensores)

Resultado de la prueba con lista vacía: ([], 0.0)
